Notebook explaining step by step how we build the `compute_likelihood_map` function.

In [ ]:
import retinoto_py as fovea
args = fovea.Params(do_fovea=True)
# args

# Likelihood map on a test image

In [ ]:
import torch
from torchvision.io import read_image

true_label = 'leopard'
image_url = './images/leopard.jpg'
# image_url = './images/jaguar.jpg'

# true_label = 'tree_frog'
# image_url = './images/frog.jpg'

alpha = 1.
s_max = 100
import cmocean
cmap = cmocean.cm.haline

full_image = read_image(image_url)/255.
print(f"{type(full_image) = }, {full_image.dtype = }, {full_image.shape = }, {full_image.min()=}, {full_image.max()=}")



In [ ]:
image_size_full = 224
image_size_full = 400

from torchvision.transforms.functional import InterpolationMode, resize

three, H, W = full_image.shape
if max((H, W)) > image_size_full:
    full_image = resize(full_image, image_size_full, interpolation=InterpolationMode.BILINEAR, antialias=True)
print(f"{type(full_image) = }, {full_image.dtype = }, {full_image.shape = }")


In [ ]:
torch.rand(2, 3, 4)

In [ ]:
full_image[:, 0, 0]

Loading the test image

In [ ]:
full_image_np = torch.movedim(full_image, (1, 2, 0), (0, 1, 2)).numpy()
fig, ax = fovea.plt.subplots()
ax.imshow(full_image_np)
# ax.set_xticks([])
# ax.set_yticks([])
fig.set_facecolor(color='white')

In [ ]:
preprocess = fovea.get_preprocess(args, do_augment=False)
# pil_image = fovea.TF.to_pil_image(full_image)
out = preprocess(full_image)
print(f"{type(out) = }, {out.dtype = }, {out.shape = }")

### testing fixate

In [ ]:
# image = image.squeeze()
# fig, ax = fovea.plt.subplots()
# image_np = torch.movedim(image, (1, 2, 0), (0, 1, 2)).cpu().numpy()
# ax.imshow(image_np)

In [ ]:
image_fix = fovea.fixate(full_image, 100, 345, 150)
full_image.shape, image_fix.shape

In [ ]:
0 <= 40 < 200

In [ ]:
fig, ax = fovea.plt.subplots()
image_np = torch.movedim(image_fix, (1, 2, 0), (0, 1, 2)).cpu().numpy()
ax.imshow(image_np)

In [ ]:
image_fix = fovea.fixate(full_image, 300, 100, 150)
fig, ax = fovea.plt.subplots()
image_np = torch.movedim(image_fix, (1, 2, 0), (0, 1, 2)).cpu().numpy()
ax.imshow(image_np)

## making a grid

First, we will define a set of fixation points as grids, that is list of horizontal and vertical coordinates. We explore different strategies.

### first strategy: valid boxes

In [ ]:
import numpy as np
resolution = (15, 23) # size of the fixation grid
size_ratio = 0.3 # how much of the image to use relative to radius

three, H, W = full_image.shape
assert three == 3

max_size = np.max((H, W))
min_size = np.min((H, W))
box_size = int(min_size*size_ratio)

if H < W:
    shift = (0, (W-H)/2)
else:
    shift = ((H-W)/2, 0)
min_size, max_size, shift[0], shift[1], box_size

In [ ]:
pos_h = np.linspace(shift[0]+box_size/2, min_size+shift[0]-box_size/2, resolution[0], endpoint=True)
pos_w = np.linspace(shift[1]+box_size/2, min_size+shift[1]-box_size/2, resolution[1], endpoint=True)
pos_h, pos_w

In [ ]:
pos_H, pos_W = np.meshgrid(pos_h, pos_w)

pos_H, pos_W = pos_H.ravel(), pos_W.ravel()
pos_H.shape, pos_W.shape

In [ ]:
fig, ax = fovea.plt.subplots()
ax.imshow(full_image_np)
ax.scatter(pos_W, pos_H, s=s_max, alpha=alpha)
ax.set_xticks([])
ax.set_yticks([])  
fig.set_facecolor(color='white')

### second strategy: more brutal

In [ ]:
aspect_ratio = H/W
N_fixations = np.prod(resolution)
resolution, N_fixations

In [ ]:
resolution = (int(np.sqrt(N_fixations*aspect_ratio)), int(np.sqrt(N_fixations/aspect_ratio)))
N_fixations = np.prod(resolution)
resolution, N_fixations

In [ ]:
pos_h = np.linspace(0, H, resolution[0]+2, endpoint=True)[1:-1]
pos_w = np.linspace(0, W, resolution[1]+2, endpoint=True)[1:-1]
pos_h, pos_w

In [ ]:
pos_H, pos_W = np.meshgrid(pos_h, pos_w)
pos_H.shape, pos_W.shape

In [ ]:
pos_H, pos_W = pos_H.ravel(), pos_W.ravel()

### second strategy: a regular grid with a hexagonal twist


In [ ]:
resolution

In [ ]:
pos_h = np.linspace(0, H, resolution[0]+2, endpoint=True)[1:-1]
pos_w = np.linspace(0, W, resolution[1]+2, endpoint=True)[1:-1]
pos_h, pos_w

In [ ]:
pos_H, pos_W = np.meshgrid(pos_h, pos_w)
if resolution[0]<=resolution[1]:
    delta = (pos_h[1]-pos_h[0])/4
    pos_H[::2] += delta
    pos_H[1::2] -= delta
else:
    delta = (pos_w[1]-pos_w[0])/4
    pos_W[::2] += delta
    pos_W[1::2] -= delta
pos_H.shape, pos_W.shape

In [ ]:
pos_H, pos_W = pos_H.ravel(), pos_W.ravel()
pos_H.shape, pos_W.shape

In [ ]:
fig, ax = fovea.plt.subplots()
ax.imshow(full_image_np)
ax.scatter(pos_W, pos_H, s=s_max, alpha=alpha)
# ax.set_xticks([])
# ax.set_yticks([])  
ax.set_xlim(0, W)
ax.set_ylim(H, 0)  
fig.set_facecolor(color='white')

### all in one function `def get_positions`

In [ ]:
pos_H, pos_W = fovea.get_positions(H, W, resolution=resolution)

In [ ]:
fig, ax = fovea.plt.subplots()
ax.imshow(full_image_np)
# ax.set_xticks([])
ax.scatter(pos_W.ravel(), pos_H.ravel(), s=s_max, alpha=alpha)
# ax.set_yticks([])  
fig.set_facecolor(color='white')

## making fixations on the positions of the grid

In [ ]:
preprocess = fovea.get_preprocess(args, do_augment=False)
out = preprocess(full_image)
print(f"{type(out) = }, {out.dtype = }, {out.shape = }")

In [ ]:
box_size, full_image.shape, out.shape

In [ ]:
image_size = max((box_size, args.image_size))
image_size

### using the `fovea.fixate` function

In [ ]:
%%timeit
preprocess(fovea.fixate(full_image, H//2, W//2, box_size))


In [ ]:
N_fixations = len(pos_H)
gaze_images = torch.empty((N_fixations, 3, box_size, box_size))
for i_fixation, (h, w) in enumerate(zip(pos_H.ravel(), pos_W.ravel())):
    h, w = int(h), int(w) 
    angle = np.random.rand()*360-180
    gaze_images[i_fixation, ...] = fovea.fixate(full_image, h, w, box_size, angle=angle)

In [ ]:
gaze_images.min(), gaze_images.max()

In [ ]:
fig, ax = fovea.imshow(gaze_images[0:(resolution[0]*3), ...], im_mean=0, im_std=1, nrow=resolution[0])

In [ ]:
fig, ax = fovea.imshow(gaze_images[(resolution[0]*6):(resolution[0]*9), ...], im_mean=0, im_std=1, nrow=resolution[0])


In [ ]:
gaze_images.shape, box_size

### preprocess the batch

In [ ]:
preprocess = fovea.get_preprocess(args, do_augment=False)
gaze_image_preprocessed = preprocess(gaze_images)

In [ ]:
fig, ax = fovea.imshow(gaze_image_preprocessed[(resolution[0]*6):(resolution[0]*9), ...], nrow=resolution[0])


### process the batch

In [ ]:
dataset = 'bbox'
model_filename = args.data_cache / f'32_fovea_model_name={args.model_name}_dataset={dataset}.pth'
model = fovea.load_model(args, model_filename=model_filename)
model_filename

In [ ]:
import torch.nn.functional as nnf

with torch.no_grad():
    gaze_image_preprocessed = gaze_image_preprocessed.to(args.device)
    probas = nnf.softmax(model(gaze_image_preprocessed), dim=1).cpu()
    # probas = nnf.sigmoid(model(gaze_image_preprocessed)).cpu()


In [ ]:
probas.shape, probas.min(), probas.max(), probas.mean()

In [ ]:
idx_to_label = fovea.get_idx_to_label(args)

In [ ]:
proba_max, preds = torch.max(probas, dim=1)
# detect = (preds == labels.data).cpu().numpy()
for proba, pred in zip(proba_max, preds):
    print(f' {idx_to_label[pred]} @ Pr={proba.item():.3f}', end ='\t')

In [ ]:
true_label

In [ ]:
label2idx = fovea.get_label_to_idx(args)
label2idx[true_label]

In [ ]:
fig, ax = fovea.plt.subplots()
ax.imshow(full_image_np)
# ax.set_xticks([])
proba_label = probas[:, label2idx[true_label]]
idx_max = proba_label.argmax()

scatter = ax.scatter(pos_W, pos_H, s=proba_label*s_max, c=proba_label, alpha=alpha, edgecolors='none', cmap=cmap, vmin=0, vmax=1)
print('Min proba_label =', min(proba_label))
print(f'Max proba_label ={max(proba_label).item():.3f}')

ax.scatter(pos_W[idx_max], pos_H[idx_max], s=proba_label[idx_max]*s_max, marker='*', c='red', alpha=alpha)
fig.colorbar(scatter, ax=ax)  # Add colorbar
# ax.scatter(500, 100, s=1000, c='r')
# ax.set_yticks([])  
fig.set_facecolor(color='white')

In [ ]:
proba_label.shape, pos_W.ravel().shape, idx_max

## as a function `fovea.compute_likelihood_map`

In [ ]:
N_fixations, N_batch = 100, 32

In [ ]:
for idx in np.arange(0, N_fixations, N_batch):
    print(idx, np.min((idx+N_batch, N_fixations)))

### comparing mappings

In [ ]:
# data_set_type = 'bbox'
# model_name = 'resnet101'


In [ ]:
three, H, W = full_image.shape
assert three == 3

pos_H, pos_W = fovea.get_positions(H, W, resolution=resolution)
probas = fovea.compute_likelihood_map(args, model, full_image, pos_H, pos_W, size_ratio=size_ratio)

proba = proba.cpu()

In [ ]:
probas.shape, pos_H.shape

In [ ]:
proba_label = probas[:, label2idx[true_label]].cpu()
proba_label

In [ ]:
idx_max = proba_label.argmax()
idx_max, pos_W[idx_max], pos_H[idx_max]

In [ ]:
def plot_map(size_ratio):
    pos_H, pos_W = fovea.get_positions(H, W, resolution=resolution)
    probas = fovea.compute_likelihood_map(args, model, full_image, pos_H, pos_W, size_ratio=size_ratio, do_softmax=True)
    probas = probas.cpu()
    proba_label = probas[:, label2idx[true_label]]
    idx_max = proba_label.argmax()

    fig, ax = fovea.plt.subplots()
    ax.imshow(full_image_np)

    scatter = ax.scatter(pos_W, pos_H, s=proba_label*s_max, c=proba_label, alpha=alpha, edgecolors='none', cmap=cmap, vmin=0, vmax=1)
    print(f'Max proba_label ={max(proba_label).item():.3f}')

    idx_max = proba_label.argmax()

    ax.scatter(pos_W[idx_max], pos_H[idx_max], s=proba_label[idx_max]*s_max, marker='*', c='red', alpha=alpha)
    fig.colorbar(scatter, ax=ax)  # Add colorbar

    fig.set_facecolor(color='white')
    return fig, ax

fig, ax = plot_map(size_ratio=size_ratio)


In [ ]:
for size_ratio in [.1, .2, .5, .618, .8]:
    print(50 * "=")
    print(f"size_ratio = {size_ratio}")
    fig, ax = plot_map(size_ratio=size_ratio)
    fovea.plt.show()

### combining different mappings

the response should be rotation and zoom invariant, so we can combine different mappings with different angles and size ratios

In [ ]:
pos_H.shape, proba_label.shape

#### combining multiple scale ratios - averaging


In [ ]:
def plot_map(size_ratios):
    pos_H, pos_W = fovea.get_positions(H, W, resolution=resolution)
    proba_label = None

    for size_ratio in size_ratios:
        probas = fovea.compute_likelihood_map(args, model, full_image, pos_H, pos_W, size_ratio=size_ratio, do_softmax=True)
        probas = probas.cpu()
        proba_label_ = probas[:, label2idx[true_label]]
        if proba_label is None:
            proba_label = proba_label_
        else:
            proba_label += proba_label_
    proba_label = proba_label / len(size_ratios)
    
    idx_max = proba_label.argmax()

    fig, ax = fovea.plt.subplots()
    ax.imshow(full_image_np)

    scatter = ax.scatter(pos_W, pos_H, s=proba_label*s_max, c=proba_label, alpha=alpha, edgecolors='none', cmap=cmap, vmin=0, vmax=1)
    print(f'Max proba_label ={max(proba_label).item():.3f}')

    idx_max = proba_label.argmax()

    ax.scatter(pos_W[idx_max], pos_H[idx_max], s=proba_label[idx_max]*s_max, marker='*', c='red', alpha=alpha)
    fig.colorbar(scatter, ax=ax)  # Add colorbar

    fig.set_facecolor(color='white')
    return fig, ax


# size_ratios = [.1, .2, .5, .618, .8]
size_ratios = np.linspace(0.1, 0.9, 9)
fig, ax = plot_map(size_ratios=size_ratios)


#### combining multiple scale ratios and angles - maximum


In [ ]:
def plot_map(size_ratios, angles):
    pos_H, pos_W = fovea.get_positions(H, W, resolution=resolution)
    proba_label, proba_label_max = None, 0.
    size_ratio_max = size_ratios[0]
    for size_ratio in size_ratios:
        for angle in angles:
            probas = fovea.compute_likelihood_map(args, model, full_image, pos_H, pos_W, size_ratio=size_ratio, angle=angle, do_softmax=True)
            probas = probas.cpu()
            proba_label_ = probas[:, label2idx[true_label]]

            if proba_label_.max() > proba_label_max:
                proba_label_max = proba_label_.max()
                size_ratio_max = size_ratio

            if proba_label is None:
                proba_label = proba_label_
            else:
                proba_label = torch.maximum(proba_label, proba_label_)

    print('size_ratio_max', size_ratio_max)

    idx_max = proba_label.argmax()

    fig, ax = fovea.plt.subplots()
    ax.imshow(full_image_np)

    scatter = ax.scatter(pos_W, pos_H, s=proba_label*s_max, c=proba_label, alpha=alpha, edgecolors='none', cmap=cmap, vmin=0, vmax=1)
    print(f'Max proba_label ={max(proba_label).item():.3f}')

    idx_max = proba_label.argmax()

    ax.scatter(pos_W[idx_max], pos_H[idx_max], s=proba_label[idx_max]*s_max, marker='*', c='red', alpha=alpha)
    fig.colorbar(scatter, ax=ax)  # Add colorbar

    fig.set_facecolor(color='white')
    return fig, ax

angles = [0, 45, 90, 135, 180]
fig, ax = plot_map(size_ratios=size_ratios, angles=angles)

#### combining multiple scale ratios and angles - averaging


In [ ]:
def plot_map(size_ratios, angles):
    pos_H, pos_W = fovea.get_positions(H, W, resolution=resolution)
    proba_label, proba_label_max = None, 0.
    size_ratio_max = size_ratios[0]
    for size_ratio in size_ratios:
        for angle in angles:
            probas = fovea.compute_likelihood_map(args, model, full_image, pos_H, pos_W, size_ratio=size_ratio, angle=angle, do_softmax=True)
            probas = probas.cpu()
            proba_label_ = probas[:, label2idx[true_label]]

            if proba_label_.max() > proba_label_max:
                proba_label_max = proba_label_.max()
                size_ratio_max = size_ratio

            if proba_label is None:
                proba_label = proba_label_
            else:
                proba_label += proba_label_
    proba_label = proba_label / len(size_ratios) / len(angles)
    
    idx_max = proba_label.argmax()

    fig, ax = fovea.plt.subplots()
    ax.imshow(full_image_np)

    scatter = ax.scatter(pos_W, pos_H, s=proba_label*s_max, c=proba_label, alpha=alpha, edgecolors='none', cmap=cmap, vmin=0, vmax=1)
    print(f'Max proba_label ={max(proba_label).item():.3f}')

    idx_max = proba_label.argmax()

    ax.scatter(pos_W[idx_max], pos_H[idx_max], s=proba_label[idx_max]*s_max, marker='*', c='red', alpha=alpha)
    fig.colorbar(scatter, ax=ax)  # Add colorbar

    fig.set_facecolor(color='white')
    return fig, ax


# size_ratios = [.1, .2, .5, .618, .8]
# size_ratios = np.linspace(0.1, 0.9, 9)
fig, ax = plot_map(size_ratios=size_ratios, angles=angles)

#### testing different lengths of angles and size ratios


In [ ]:
fig, ax = plot_map(size_ratios=size_ratios, angles=[0, 60, 120])

In [ ]:
# fig, ax = plot_map(size_ratios=np.linspace(0.3, 0.9, 5), angles=angles)

In [ ]:
# fig, ax = plot_map(size_ratios=np.linspace(0.1, 0.6, 5), angles=angles)

In [ ]:
fig, ax = plot_map(size_ratios=[0.1, 0.2, 0.4, 0.6, 0.9], angles=angles)

In [ ]:
fig, ax = plot_map(size_ratios=size_ratios, angles=[0, 90, 180])

In [ ]:
# fig, ax = plot_map(size_ratios=size_ratios, angles=[0, 60, -30])

Voilà !